# AgriSmart AI — EfficientNet-B2 Training on Colab GPU

**Complete self-contained training notebook.**

### Before running:
1. `Runtime → Change runtime type → T4 GPU` (free tier)
2. Run all cells top-to-bottom
3. After training, download `agrismart_best.pth` and `classes.json` from the Files panel

### What this notebook does:
- Downloads PlantVillage dataset from Hugging Face
- Applies the 38→28 class mapping
- Creates an 80/10/10 leakage-safe train/val/test split
- Trains EfficientNet-B2 (2-stage: warmup + fine-tune)
- Saves best model by validation Macro-F1
- Downloads result files automatically

## Cell 1 — Check GPU

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU found. Go to Runtime > Change runtime type > GPU')

## Cell 2 — Install Dependencies

In [ ]:
!pip install -q timm datasets scikit-learn tqdm seaborn matplotlib

## Cell 3 — Configuration

In [ ]:
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
ROOT          = Path('/content/agrismart')
RAW_DIR       = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
TRAIN_DIR     = PROCESSED_DIR / 'train'
VAL_DIR       = PROCESSED_DIR / 'val'
TEST_DIR      = PROCESSED_DIR / 'test'
MODELS_DIR    = ROOT / 'models'
EVAL_DIR      = ROOT / 'evaluation' / 'results'

for d in [TRAIN_DIR, VAL_DIR, TEST_DIR, MODELS_DIR, EVAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODELS_DIR / 'agrismart_best.pth'
LAST_MODEL_PATH = MODELS_DIR / 'agrismart_last.pth'
CLASSES_JSON    = MODELS_DIR / 'classes.json'
EXPERIMENT_LOG  = ROOT / 'experiments.csv'

# ── Model ──────────────────────────────────────────────────────────────────────
MODEL_NAME  = 'efficientnet_b2'
PRETRAINED  = True
IMAGE_SIZE  = 260

# ── Training ───────────────────────────────────────────────────────────────────
SEED                  = 42
BATCH_SIZE            = 32
NUM_EPOCHS            = 25
LEARNING_RATE         = 1e-4
WEIGHT_DECAY          = 1e-4
WARMUP_EPOCHS         = 3     # freeze backbone for first N epochs
EARLY_STOPPING_PATIENCE = 7
LR_SCHEDULER          = 'cosine'
OPTIMIZER             = 'adamw'
USE_AMP               = True
USE_CLASS_WEIGHTS     = True
AUGMENTATION_LEVEL    = 'medium'
NUM_WORKERS           = 4

# ── Normalization (ImageNet stats) ──────────────────────────────────────────────
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

# ── Confidence ────────────────────────────────────────────────────────────────
CONFIDENCE_HIGH   = 0.80
CONFIDENCE_MEDIUM = 0.60

# ── 28 Official Development Classes ────────────────────────────────────────────
DEVELOPMENT_CLASSES = [
    'Apple — Apple Scab',
    'Apple — Healthy',
    'Apple — Cedar Apple Rust',
    'Blueberry — Healthy',
    'Cherry — Healthy',
    'Corn — Cercospora Leaf Spot / Gray Leaf Spot',
    'Corn — Common Rust',
    'Corn — Northern Leaf Blight',
    'Grape — Black Rot',
    'Grape — Healthy',
    'Peach — Healthy',
    'Bell Pepper — Bacterial Spot',
    'Bell Pepper — Healthy',
    'Potato — Early Blight',
    'Potato — Late Blight',
    'Raspberry — Healthy',
    'Soybean — Healthy',
    'Squash — Powdery Mildew',
    'Strawberry — Healthy',
    'Tomato — Bacterial Spot',
    'Tomato — Early Blight',
    'Tomato — Late Blight',
    'Tomato — Leaf Mold',
    'Tomato — Septoria Leaf Spot',
    'Tomato — Spider Mites / Two-Spotted Spider Mite',
    'Tomato — Tomato Yellow Leaf Curl Virus',
    'Tomato — Tomato Mosaic Virus',
    'Tomato — Healthy',
]

print(f'Config loaded. Classes: {len(DEVELOPMENT_CLASSES)}')

## Cell 4 — 38→28 Class Mapping Table

In [ ]:
# Exact mapping from PlantVillage folder names → 28 development class labels
CLASS_MAPPING = {
    # Apple
    'Apple___Apple_scab':                                  'Apple — Apple Scab',
    'Apple___Black_rot':                                   None,   # excluded
    'Apple___Cedar_apple_rust':                            'Apple — Cedar Apple Rust',
    'Apple___healthy':                                     'Apple — Healthy',
    # Blueberry
    'Blueberry___healthy':                                 'Blueberry — Healthy',
    # Cherry
    'Cherry_(including_sour)___Powdery_mildew':            None,   # excluded
    'Cherry_(including_sour)___healthy':                   'Cherry — Healthy',
    # Corn
    'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot':  'Corn — Cercospora Leaf Spot / Gray Leaf Spot',
    'Corn_(maize)___Common_rust_':                         'Corn — Common Rust',
    'Corn_(maize)___Northern_Leaf_Blight':                 'Corn — Northern Leaf Blight',
    'Corn_(maize)___healthy':                              None,   # excluded
    # Grape
    'Grape___Black_rot':                                   'Grape — Black Rot',
    'Grape___Esca_(Black_Measles)':                        None,   # excluded
    'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)':          None,   # excluded
    'Grape___healthy':                                     'Grape — Healthy',
    # Orange
    'Orange___Haunglongbing_(Citrus_greening)':            None,   # excluded
    # Peach
    'Peach___Bacterial_spot':                              None,   # excluded
    'Peach___healthy':                                     'Peach — Healthy',
    # Pepper
    'Pepper,_bell___Bacterial_spot':                       'Bell Pepper — Bacterial Spot',
    'Pepper,_bell___healthy':                              'Bell Pepper — Healthy',
    # Potato
    'Potato___Early_blight':                               'Potato — Early Blight',
    'Potato___Late_blight':                                'Potato — Late Blight',
    'Potato___healthy':                                    None,   # excluded
    # Raspberry
    'Raspberry___healthy':                                 'Raspberry — Healthy',
    # Soybean
    'Soybean___healthy':                                   'Soybean — Healthy',
    # Squash
    'Squash___Powdery_mildew':                             'Squash — Powdery Mildew',
    # Strawberry
    'Strawberry___Leaf_scorch':                            None,   # excluded
    'Strawberry___healthy':                                'Strawberry — Healthy',
    # Tomato
    'Tomato___Bacterial_spot':                             'Tomato — Bacterial Spot',
    'Tomato___Early_blight':                               'Tomato — Early Blight',
    'Tomato___Late_blight':                                'Tomato — Late Blight',
    'Tomato___Leaf_Mold':                                  'Tomato — Leaf Mold',
    'Tomato___Septoria_leaf_spot':                         'Tomato — Septoria Leaf Spot',
    'Tomato___Spider_mites Two-spotted_spider_mite':       'Tomato — Spider Mites / Two-Spotted Spider Mite',
    'Tomato___Target_Spot':                                None,   # excluded
    'Tomato___Tomato_Yellow_Leaf_Curl_Virus':              'Tomato — Tomato Yellow Leaf Curl Virus',
    'Tomato___Tomato_mosaic_virus':                        'Tomato — Tomato Mosaic Virus',
    'Tomato___healthy':                                    'Tomato — Healthy',
}

print(f'Total PlantVillage folders in mapping: {len(CLASS_MAPPING)}')
included = {k: v for k, v in CLASS_MAPPING.items() if v is not None}
excluded = {k for k, v in CLASS_MAPPING.items() if v is None}
print(f'Mapped to development classes: {len(included)}')
print(f'Excluded (None): {len(excluded)}')

## Cell 5 — Download PlantVillage Dataset

In [ ]:
import os
import shutil
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

print('Downloading PlantVillage from Hugging Face...')
print('This will take a few minutes...')

# Clear raw dir if re-running
if RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)

ds = load_dataset('mohanty/PlantVillage', split='train', trust_remote_code=False)
print(f'Downloaded {len(ds)} images')
print(f'Features: {ds.features}')

# Determine label column
label_col = None
for col in ['label', 'labels', 'disease', 'category']:
    if col in ds.features:
        label_col = col
        break
if label_col is None:
    label_col = [c for c in ds.features if c != 'image'][0]

print(f'Label column: {label_col}')

# Get class names
if hasattr(ds.features[label_col], 'names'):
    id2label = {i: name for i, name in enumerate(ds.features[label_col].names)}
else:
    id2label = {v: v for v in set(ds[label_col])}

print(f'Number of raw classes: {len(id2label)}')
print('\nSample class names:')
for k, v in list(id2label.items())[:5]:
    print(f'  {k}: {v}')

## Cell 6 — Save Raw Images to Disk by Class Folder

In [ ]:
print('Saving raw images to disk...')
saved = 0
skipped = 0

for sample in tqdm(ds, desc='Saving images'):
    label_id = sample[label_col]
    class_name = id2label[label_id] if isinstance(label_id, int) else label_id
    class_dir = RAW_DIR / class_name
    class_dir.mkdir(parents=True, exist_ok=True)

    img_count = len(list(class_dir.glob('*.jpg')))
    out_path = class_dir / f'{img_count:05d}.jpg'

    try:
        img = sample['image']
        if not isinstance(img, Image.Image):
            img = Image.fromarray(img)
        img.convert('RGB').save(out_path, format='JPEG', quality=95)
        saved += 1
    except Exception as e:
        skipped += 1

print(f'\nSaved: {saved} | Skipped/corrupt: {skipped}')

raw_classes = [d.name for d in sorted(RAW_DIR.iterdir()) if d.is_dir()]
print(f'Raw class folders: {len(raw_classes)}')

## Cell 7 — Apply 38→28 Class Mapping

In [ ]:
MAPPED_DIR = ROOT / 'data' / 'mapped'

if MAPPED_DIR.exists():
    shutil.rmtree(MAPPED_DIR)
MAPPED_DIR.mkdir(parents=True, exist_ok=True)

mapped_count = 0
skipped_count = 0
ambiguous = []

for raw_class_dir in sorted(RAW_DIR.iterdir()):
    if not raw_class_dir.is_dir():
        continue
    folder = raw_class_dir.name

    # Direct lookup
    dev_class = CLASS_MAPPING.get(folder)
    if dev_class is None and folder not in CLASS_MAPPING:
        # Try fuzzy: normalize underscores, case
        norm = folder.replace(' ', '_')
        for k, v in CLASS_MAPPING.items():
            if k.replace(' ', '_').lower() == norm.lower():
                dev_class = v
                break
        if dev_class is None:
            # Unmapped: check if it has a None explicit mapping
            if folder in CLASS_MAPPING:
                skipped_count += len(list(raw_class_dir.glob('*.jpg')))
                continue
            else:
                print(f'  [WARN] Unknown folder (no mapping): {folder}')
                ambiguous.append(folder)
                continue
    elif dev_class is None:
        # Explicitly excluded
        skipped_count += len(list(raw_class_dir.glob('*.jpg')))
        continue

    # Sanitize class name for filesystem
    safe_name = dev_class  # keep as-is — will be class folder name
    dest_dir = MAPPED_DIR / safe_name
    dest_dir.mkdir(parents=True, exist_ok=True)

    images = list(raw_class_dir.glob('*.jpg')) + list(raw_class_dir.glob('*.JPG')) + \
             list(raw_class_dir.glob('*.png')) + list(raw_class_dir.glob('*.PNG'))
    existing = len(list(dest_dir.glob('*.jpg')))

    for i, img_path in enumerate(images):
        dest = dest_dir / f'{existing + i:05d}.jpg'
        shutil.copy2(str(img_path), str(dest))
        mapped_count += 1

print(f'\nMapped images:  {mapped_count}')
print(f'Skipped (excluded classes): {skipped_count}')
if ambiguous:
    print(f'AMBIGUOUS folders (no mapping): {ambiguous}')

mapped_classes = [d.name for d in sorted(MAPPED_DIR.iterdir()) if d.is_dir()]
print(f'\nMapped class folders: {len(mapped_classes)}')
for cls in sorted(mapped_classes):
    count = len(list((MAPPED_DIR / cls).glob('*.jpg')))
    print(f'  {cls:<55} {count:>5}')

## Cell 8 — Create 80/10/10 Leakage-Safe Split

In [ ]:
import random
import numpy as np

random.seed(SEED)
np.random.seed(SEED)

TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10

# Clear existing splits
for split_dir in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    if split_dir.exists():
        shutil.rmtree(split_dir)
    split_dir.mkdir(parents=True, exist_ok=True)

split_stats = {'train': {}, 'val': {}, 'test': {}}
total = {'train': 0, 'val': 0, 'test': 0}

for cls_dir in sorted(MAPPED_DIR.iterdir()):
    if not cls_dir.is_dir():
        continue
    cls_name = cls_dir.name
    images = sorted(cls_dir.glob('*.jpg'))
    random.shuffle(images)

    n = len(images)
    n_train = max(1, int(n * TRAIN_RATIO))
    n_val   = max(1, int(n * VAL_RATIO))
    n_test  = n - n_train - n_val
    if n_test < 1:
        n_test = 1
        n_train = n - n_val - n_test

    splits_imgs = {
        'train': images[:n_train],
        'val':   images[n_train:n_train + n_val],
        'test':  images[n_train + n_val:],
    }

    for split_name, split_imgs in splits_imgs.items():
        dest_dir = (TRAIN_DIR if split_name == 'train' else
                    VAL_DIR   if split_name == 'val'   else TEST_DIR) / cls_name
        dest_dir.mkdir(parents=True, exist_ok=True)
        for img in split_imgs:
            shutil.copy2(str(img), str(dest_dir / img.name))
        split_stats[split_name][cls_name] = len(split_imgs)
        total[split_name] += len(split_imgs)

print('Split complete:')
print(f'  Train: {total["train"]:>6}')
print(f'  Val:   {total["val"]:>6}')
print(f'  Test:  {total["test"]:>6}')
print(f'  Total: {sum(total.values()):>6}')

# Verify all 28 classes in all splits
for split_name in ['train', 'val', 'test']:
    missing = [c for c in DEVELOPMENT_CLASSES if c not in split_stats[split_name]]
    print(f'  {split_name}: {len(split_stats[split_name])} classes, {len(missing)} missing')

## Cell 9 — Define Transforms

In [ ]:
import io
import torchvision.transforms as T

class AddGaussianNoise:
    def __init__(self, std=0.03, p=0.2):
        self.std = std
        self.p = p
    def __call__(self, tensor):
        if random.random() < self.p:
            return torch.clamp(tensor + torch.randn_like(tensor) * self.std, 0., 1.)
        return tensor

class JPEGCompressionDegradation:
    def __init__(self, quality_min=40, quality_max=85, p=0.25):
        self.quality_min = quality_min
        self.quality_max = quality_max
        self.p = p
    def __call__(self, img):
        if random.random() < self.p:
            buf = io.BytesIO()
            quality = random.randint(self.quality_min, self.quality_max)
            img.save(buf, format='JPEG', quality=quality)
            buf.seek(0)
            return Image.open(buf).convert('RGB')
        return img

train_transform = T.Compose([
    JPEGCompressionDegradation(quality_min=40, quality_max=85, p=0.25),
    T.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.2),
    T.RandomRotation(degrees=20),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.08),
    T.GaussianBlur(kernel_size=3, sigma=(0.1, 1.2)),
    T.ToTensor(),
    AddGaussianNoise(std=0.03, p=0.2),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
    T.RandomErasing(p=0.2, scale=(0.02, 0.1), ratio=(0.3, 3.3), value=0),
])

val_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

print('Transforms defined.')
print(f'  Image size: {IMAGE_SIZE}x{IMAGE_SIZE}')
print(f'  Augmentation level: medium')

## Cell 10 — Build DataLoaders

In [ ]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, WeightedRandomSampler

train_ds = ImageFolder(str(TRAIN_DIR), transform=train_transform)
val_ds   = ImageFolder(str(VAL_DIR),   transform=val_transform)
test_ds  = ImageFolder(str(TEST_DIR),  transform=val_transform)

# Verify class names match development classes
class_names = train_ds.classes
print(f'Classes from ImageFolder: {len(class_names)}')

# Weighted sampler for class imbalance
class_counts = torch.zeros(len(class_names))
for _, lbl in train_ds.samples:
    class_counts[lbl] += 1
class_weights = 1.0 / (class_counts + 1e-6)
class_weights = class_weights / class_weights.sum() * len(class_names)
sample_weights = [class_weights[lbl].item() for _, lbl in train_ds.samples]
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds)} images, {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} images, {len(val_loader)} batches')
print(f'Test:  {len(test_ds)} images, {len(test_loader)} batches')
print(f'Batch size: {BATCH_SIZE}')

## Cell 11 — Build Model

In [ ]:
import timm
import torch.nn as nn
import torch.optim as optim
from torch.amp import GradScaler, autocast

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training device: {device}')

num_classes = len(class_names)
model = timm.create_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=num_classes)
model = model.to(device)

# Loss with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# AMP scaler
scaler = GradScaler('cuda', enabled=USE_AMP)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: {MODEL_NAME}')
print(f'Total params:     {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(f'Classes: {num_classes}')

## Cell 12 — Training Loop

In [ ]:
import csv
import time
import json
from datetime import datetime
from sklearn.metrics import f1_score, accuracy_score
from tqdm.auto import tqdm

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

def train_one_epoch(model, loader, criterion, optimizer, scaler, device, epoch):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, desc=f'Epoch {epoch+1} [Train]', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast('cuda', enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    accuracy  = accuracy_score(all_labels, all_preds)
    return avg_loss, macro_f1, accuracy


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, desc='[Val]', leave=False):
        images, labels = images.to(device), labels.to(device)
        with autocast('cuda', enabled=USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)
        total_loss += loss.item()
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    accuracy  = accuracy_score(all_labels, all_preds)
    return avg_loss, macro_f1, accuracy


def log_experiment(row):
    path = EXPERIMENT_LOG
    file_exists = path.exists()
    with open(path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=row.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)


# ── Main Training Loop ──────────────────────────────────────────────────────────
best_val_f1 = 0.0
epochs_no_improve = 0
experiment_id = datetime.now().strftime('%Y%m%d_%H%M%S')

print(f'Starting training | Model: {MODEL_NAME} | Epochs: {NUM_EPOCHS} | Device: {device}')
print('=' * 70)

for epoch in range(NUM_EPOCHS):
    # Two-stage training: freeze backbone during warmup
    if epoch == 0:
        print(f'Stage 1: Freezing backbone for {WARMUP_EPOCHS} warmup epochs...')
        for name, param in model.named_parameters():
            if not any(x in name for x in ['classifier', 'head', 'fc']):
                param.requires_grad = False
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  Trainable params (head only): {trainable:,}')

    elif epoch == WARMUP_EPOCHS:
        print(f'Stage 2: Unfreezing all layers (fine-tuning)...')
        for param in model.parameters():
            param.requires_grad = True
        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'  Trainable params (all): {trainable:,}')

    t0 = time.time()
    train_loss, train_f1, train_acc = train_one_epoch(
        model, train_loader, criterion, optimizer, scaler, device, epoch)
    val_loss, val_f1, val_acc = validate(model, val_loader, criterion, device)
    elapsed = time.time() - t0

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    print(
        f'Epoch {epoch+1:03d}/{NUM_EPOCHS} | '
        f'Train  Loss:{train_loss:.4f} F1:{train_f1:.4f} Acc:{train_acc:.4f} | '
        f'Val  Loss:{val_loss:.4f} F1:{val_f1:.4f} Acc:{val_acc:.4f} | '
        f'LR:{current_lr:.2e} | {elapsed:.0f}s'
    )

    # Save best model
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        epochs_no_improve = 0
        torch.save({
            'epoch':         epoch + 1,
            'model_name':    MODEL_NAME,
            'num_classes':   num_classes,
            'class_names':   class_names,
            'image_size':    IMAGE_SIZE,
            'state_dict':    model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'val_macro_f1':  val_f1,
            'val_accuracy':  val_acc,
        }, BEST_MODEL_PATH)
        print(f'  --> Best model saved (Val Macro-F1: {best_val_f1:.4f})')
    else:
        epochs_no_improve += 1

    # Save last checkpoint
    torch.save({'epoch': epoch + 1, 'state_dict': model.state_dict()}, LAST_MODEL_PATH)

    # Log
    log_experiment({
        'experiment_id': experiment_id, 'epoch': epoch+1, 'model': MODEL_NAME,
        'batch_size': BATCH_SIZE, 'lr': current_lr,
        'train_loss': round(train_loss, 4), 'train_f1': round(train_f1, 4),
        'train_acc': round(train_acc, 4),   'val_loss': round(val_loss, 4),
        'val_f1': round(val_f1, 4),         'val_acc': round(val_acc, 4),
        'best_val_f1': round(best_val_f1, 4),
    })

    # Early stopping
    if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
        print(f'\nEarly stopping after {EARLY_STOPPING_PATIENCE} epochs without improvement.')
        break

print(f'\nTraining complete. Best Val Macro-F1: {best_val_f1:.4f}')
print(f'Model saved: {BEST_MODEL_PATH}')

## Cell 13 — Save classes.json

In [ ]:
mapping = {str(i): name for i, name in enumerate(class_names)}
with open(CLASSES_JSON, 'w') as f:
    json.dump(mapping, f, indent=2)
print(f'Saved: {CLASSES_JSON}')
print('Class mapping:')
for k, v in mapping.items():
    print(f'  {k}: {v}')

## Cell 14 — Full Evaluation on Validation Set

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# Load best model
ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
eval_model = timm.create_model(ckpt['model_name'], pretrained=False, num_classes=ckpt['num_classes'])
eval_model.load_state_dict(ckpt['state_dict'])
eval_model = eval_model.to(device)
eval_model.eval()

ckpt_classes = ckpt['class_names']
print(f'Loaded checkpoint: epoch {ckpt["epoch"]}, Val Macro-F1={ckpt["val_macro_f1"]:.4f}')

# Collect predictions on val set
all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for images, labels in tqdm(val_loader, desc='Evaluating val'):
        images = images.to(device)
        with autocast('cuda', enabled=USE_AMP):
            outputs = eval_model(images)
        probs = torch.softmax(outputs, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_probs.extend(probs)
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
accuracy  = accuracy_score(all_labels, all_preds)

print(f'\n{"="*60}')
print(f'  Val Macro-F1:  {macro_f1:.4f}')
print(f'  Val Accuracy:  {accuracy:.4f}')
print(f'{"="*60}')
print()
print(classification_report(all_labels, all_preds, target_names=ckpt_classes, zero_division=0))

## Cell 15 — Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-6)

fig, ax = plt.subplots(figsize=(16, 14))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Blues',
    xticklabels=ckpt_classes, yticklabels=ckpt_classes,
    ax=ax, linewidths=0.5, annot_kws={'size': 6},
)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.set_title('Normalised Confusion Matrix — AgriSmart AI (Validation Set)', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.yticks(rotation=0, fontsize=7)
plt.tight_layout()

cm_path = EVAL_DIR / 'confusion_matrix_val.png'
plt.savefig(cm_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {cm_path}')

## Cell 16 — Error Analysis

In [ ]:
from sklearn.metrics import recall_score, precision_score
from collections import Counter

recalls    = recall_score(all_labels, all_preds, average=None, zero_division=0)
precisions = precision_score(all_labels, all_preds, average=None, zero_division=0)
f1s        = f1_score(all_labels, all_preds, average=None, zero_division=0)

class_perf = sorted(zip(ckpt_classes, recalls, precisions, f1s), key=lambda x: x[3])

print('Bottom 5 classes by F1:')
for name, rec, prec, f1 in class_perf[:5]:
    print(f'  {name:<50}  Recall:{rec:.3f}  Prec:{prec:.3f}  F1:{f1:.3f}')

print('\nTop 5 classes by F1:')
for name, rec, prec, f1 in class_perf[-5:]:
    print(f'  {name:<50}  Recall:{rec:.3f}  Prec:{prec:.3f}  F1:{f1:.3f}')

wrong_mask = all_labels != all_preds
wrong_true = all_labels[wrong_mask]
wrong_pred = all_preds[wrong_mask]
pairs = Counter(zip(wrong_true.tolist(), wrong_pred.tolist()))
print('\nTop 10 confusion pairs (true -> predicted):')
for (t, p), count in pairs.most_common(10):
    print(f'  {ckpt_classes[t]:<45} -> {ckpt_classes[p]:<45} ({count} errors)')

max_probs = all_probs.max(axis=1)
correct_mask = all_labels == all_preds
print(f'\nConfidence stats:')
print(f'  Correct  mean confidence: {max_probs[correct_mask].mean():.3f}')
print(f'  Wrong    mean confidence: {max_probs[~correct_mask].mean():.3f}')
print(f'  Low conf (<0.60): {(max_probs < 0.60).sum()} samples')
print(f'  High conf (>0.80): {(max_probs > 0.80).sum()} samples')

## Cell 17 — Save Evaluation JSON

In [ ]:
report_dict = classification_report(
    all_labels, all_preds,
    target_names=ckpt_classes,
    output_dict=True,
    zero_division=0,
)

result = {
    'split':     'val',
    'macro_f1':  round(macro_f1, 4),
    'accuracy':  round(accuracy, 4),
    'model':     MODEL_NAME,
    'epoch':     int(ckpt['epoch']),
    'per_class': {
        name: {
            'precision': round(report_dict[name]['precision'], 4),
            'recall':    round(report_dict[name]['recall'], 4),
            'f1':        round(report_dict[name]['f1-score'], 4),
            'support':   int(report_dict[name]['support']),
        }
        for name in ckpt_classes if name in report_dict
    },
}

out_path = EVAL_DIR / 'evaluation_val.json'
with open(out_path, 'w') as f:
    json.dump(result, f, indent=2)
print(f'Saved: {out_path}')
print(f'Final Val Macro-F1: {macro_f1:.4f}')

## Cell 18 — Download Results

Run this cell to download all output files to your local machine.

In [ ]:
from google.colab import files

print('Downloading output files...')

# Download model weights
if BEST_MODEL_PATH.exists():
    files.download(str(BEST_MODEL_PATH))
    print(f'Downloaded: agrismart_best.pth')

# Download class mapping
if CLASSES_JSON.exists():
    files.download(str(CLASSES_JSON))
    print(f'Downloaded: classes.json')

# Download experiment log
if EXPERIMENT_LOG.exists():
    files.download(str(EXPERIMENT_LOG))
    print(f'Downloaded: experiments.csv')

# Download evaluation JSON
eval_json = EVAL_DIR / 'evaluation_val.json'
if eval_json.exists():
    files.download(str(eval_json))
    print(f'Downloaded: evaluation_val.json')

# Download confusion matrix
cm_path = EVAL_DIR / 'confusion_matrix_val.png'
if cm_path.exists():
    files.download(str(cm_path))
    print(f'Downloaded: confusion_matrix_val.png')

print('\nDone! Place the downloaded files in:')
print('  agrismart_best.pth  ->  D:/Agrismart_AI/models/')
print('  classes.json        ->  D:/Agrismart_AI/models/')
print('  experiments.csv     ->  D:/Agrismart_AI/')
print('  evaluation_val.json ->  D:/Agrismart_AI/evaluation/results/')
print('  confusion_matrix_val.png -> D:/Agrismart_AI/evaluation/results/')